In [1]:
#%pip install country_converter

In [2]:
import pandas as pd
import json
import numpy as np
import re
import ast

from pathlib import Path

### institutionsTopicsSankeyData

In [3]:
SEC3B_PATH = Path("../data/processed/outputs/openalex_notebook_outputs/tables/sec3b_institution_macrocategory_counts.csv")
INST_PATH  = Path("../data/processed/outputs/openalex_notebook_outputs/tables/institutions.csv")

# (opzionale) per decidere l’ordine delle 10 macrocategory (se ce l'hai)
MACRO_SUMMARY_PATH = Path("../data/processed/output/macro_summary.csv")

# OUTPUT
OUT_PATH = Path("../data/processed/output/institutions_macro_matrix.csv")

# -------------------------
# Load
# -------------------------
sec3b = pd.read_csv(SEC3B_PATH)
inst  = pd.read_csv(INST_PATH)

sec3b["institution_id"] = sec3b["institution_id"].astype(str)

# Tieni solo istituzioni vere (presenti in institutions.csv)
valid_inst_ids = set(inst["institution_id"].astype(str))
sec3b = sec3b[sec3b["institution_id"].isin(valid_inst_ids)].copy()

# -------------------------
# Ordine colonne MacroCategory (10)
# -------------------------
if MACRO_SUMMARY_PATH.exists():
    macro_sum = pd.read_csv(MACRO_SUMMARY_PATH)
    # Ordina le MacroCategory per importanza globale (somma Count) -> stabile e sensato
    macro_order = (macro_sum.groupby("MacroCategory")["Count"]
                   .sum()
                   .sort_values(ascending=False)
                   .index.tolist())
else:
    # fallback: ordine per totale n_papers nel tuo sec3b
    macro_order = (sec3b.groupby("MacroCategory")["n_papers"]
                   .sum()
                   .sort_values(ascending=False)
                   .index.tolist())

# tieni solo le macro che effettivamente appaiono (di solito 10)
macro_order = [m for m in macro_order if m in set(sec3b["MacroCategory"])]
# se sono più di 10 per qualche motivo, taglia a 10
macro_order = macro_order[:10]

# -------------------------
# Pivot: institution × MacroCategory
# -------------------------
pivot = (sec3b.pivot_table(index="institution_id",
                           columns="MacroCategory",
                           values="n_papers",
                           aggfunc="sum",
                           fill_value=0)
         .reset_index())

# garantisci tutte le 10 colonne (anche se qualche macro manca in pivot)
for m in macro_order:
    if m not in pivot.columns:
        pivot[m] = 0

pivot = pivot[["institution_id"] + macro_order]

# totale
pivot["total"] = pivot[macro_order].sum(axis=1)

# -------------------------
# Join info istituzione
# -------------------------
inst_small = inst[["institution_id", "display_name", "country_name", "world_region"]].copy()
out = pivot.merge(inst_small, on="institution_id", how="left")

# riordina colonne: nome + macro + total
out = out[["institution_id", "display_name", "country_name", "world_region"] + macro_order + ["total"]]

# ordina
out = out.sort_values("total", ascending=False)

# -------------------------
# Rinomina MacroCategory -> Mcat0..Mcat9 (come volevi)
# + salva anche un file mapping per sapere a cosa corrisponde ogni Mcat
# -------------------------
rename_map = {macro_order[i]: f"Mcat{i}" for i in range(len(macro_order))}
out_renamed = out.rename(columns=rename_map)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out_renamed.to_csv(OUT_PATH, index=False)

# mapping (super utile per DV)
MAP_PATH = OUT_PATH.with_name("institutions_macro_matrix_mapping.csv")
pd.DataFrame({
    "Mcat": [f"Mcat{i}" for i in range(len(macro_order))],
    "MacroCategory": macro_order
}).to_csv(MAP_PATH, index=False)

print("✅ Wrote:", OUT_PATH)
print("✅ Mapping:", MAP_PATH)
out_renamed.head(10)


✅ Wrote: ../data/processed/output/institutions_macro_matrix.csv
✅ Mapping: ../data/processed/output/institutions_macro_matrix_mapping.csv


,institution_id,display_name,country_name,world_region,Mcat0,Mcat1,Mcat2,Mcat3,Mcat4,Mcat5,Mcat6,Mcat7,Mcat8,Mcat9,total
457,I223532165,University of Utah,United States,Americas,68,24,6,14,6,1,4,19,3,1,146
1090,I84218800,"University of California, Davis",United States,Americas,46,18,3,9,7,14,7,8,5,2,119
419,I200769079,Hong Kong University of Science and Technology,Hong Kong,Asia,43,8,22,5,6,15,0,2,0,3,104
1117,I889458895,University of Hong Kong,Hong Kong,Asia,39,8,22,5,6,16,1,2,0,3,102
244,I145847075,TU Wien,Austria,Europe,51,6,13,13,6,4,5,0,1,0,99
1017,I59553526,Stony Brook University,United States,Americas,54,7,2,11,6,10,0,0,1,1,92
1,I100066346,University of Stuttgart,Germany,Europe,31,20,10,4,8,6,10,1,1,0,91
148,I130701444,Georgia Institute of Technology,United States,Americas,51,3,10,2,8,9,3,2,1,1,90
1065,I76130692,Zhejiang University,China,Asia,40,21,10,5,5,6,0,0,0,2,89
383,I189712700,University of Konstanz,Germany,Europe,42,4,9,3,13,2,1,2,0,3,79


In [4]:
# =========================
# INPUT (CSV matrix + mapping)
# =========================
MATRIX_CSV = Path("../data/processed/output/institutions_macro_matrix.csv")
MAP_CSV    = Path("../data/processed/output/institutions_macro_matrix_mapping.csv")

# OUTPUT JS
OUT_JS = Path("../site/data/research/institutionsTopicsData.js")

# PARAMS
TOP_N = 20        # oppure 20
MIN_VALUE = 1     # link solo se >= 1 (puoi mettere 2/3 per ridurre densità)

# =========================
# LOAD
# =========================
df = pd.read_csv(MATRIX_CSV)
mp = pd.read_csv(MAP_CSV)

# colonne Mcat in ordine
mcat_cols = mp.sort_values("Mcat")["Mcat"].tolist()
mcat_to_name = dict(zip(mp["Mcat"], mp["MacroCategory"]))

# prendi top N istituzioni
df_top = df.sort_values("total", ascending=False).head(TOP_N).copy()

# =========================
# region mapping (compat con sankey.js)
# =========================
def map_region(world_region):
    if not isinstance(world_region, str) or not world_region.strip():
        return "Other"
    wr = world_region.strip()
    if wr == "Americas":
        return "North America"
    if wr in {"Europe", "Asia", "North America"}:
        return wr
    return "Other"

# =========================
# NODES
# =========================
nodes = []

# institution nodes (left)
for _, r in df_top.iterrows():
    nodes.append({
        "id": r["display_name"],
        "type": "institution",
        "region": map_region(r.get("world_region"))
    })

# topic nodes (right) = MacroCategory (10)
topic_names = [mcat_to_name[c] for c in mcat_cols]
for t in topic_names:
    nodes.append({
        "id": t,
        "type": "topic",
        "category": t
    })

# =========================
# LINKS
# =========================
links = []
strongest = {"institution": None, "topic": None, "papers": 0}

for _, r in df_top.iterrows():
    inst_name = r["display_name"]
    for c in mcat_cols:
        v = int(r.get(c, 0))
        if v >= MIN_VALUE:
            topic = mcat_to_name[c]
            links.append({"source": inst_name, "target": topic, "value": v})
            if v > strongest["papers"]:
                strongest = {"institution": inst_name, "topic": topic, "papers": v}

# =========================
# topicColors (1 colore per topic)
# =========================
palette = [
    "#3b82f6", "#10b981", "#f59e0b", "#ef4444", "#8b5cf6",
    "#06b6d4", "#84cc16", "#f97316", "#ec4899", "#64748b"
]
topicColors = {t: palette[i % len(palette)] for i, t in enumerate(topic_names)}

# =========================
# Stats (semplici + opzionali)
# =========================
institutionsTopicsData = {"nodes": nodes, "links": links}

institutionsTopicsStats = {
    "topInstitutions": int(TOP_N),
    "topTopics": int(len(topic_names)),
    "totalConnections": int(len(links)),
    "strongestConnection": strongest,
}

# =========================
# WRITE JS (newlines vere, no \\n)
# =========================
OUT_JS.parent.mkdir(parents=True, exist_ok=True)

lines = [
    "/**",
    " * data/research/institutionsTopicsData.js",
    f" * Sankey diagram data connecting top {TOP_N} institutions to MacroCategory topics",
    " * AUTO-GENERATED from institutions_macro_matrix.csv",
    " */",
    "",
    "export const institutionsTopicsData = " + json.dumps(institutionsTopicsData, ensure_ascii=False, indent=2) + ";",
    "",
    "export const institutionsTopicsStats = " + json.dumps(institutionsTopicsStats, ensure_ascii=False, indent=2) + ";",
    "",
    "export const topicColors = " + json.dumps(topicColors, ensure_ascii=False, indent=2) + ";",
    "",
]

OUT_JS.write_text("\n".join(lines), encoding="utf-8")
print("✅ Wrote:", OUT_JS)
print("Nodes:", len(nodes), "Links:", len(links))
print("Strongest:", strongest)


✅ Wrote: ../site/data/research/institutionsTopicsData.js
Nodes: 30 Links: 169
Strongest: {'institution': 'University of Utah', 'topic': 'Dimensionality Reduction & Multivariate Plots', 'papers': 68}


In [10]:
SRC = Path("../data/processed/outputs/openalex_notebook_outputs/tables/sec3a_institutions_map.csv")
OUT = Path("../data/processed/outputs/openalex_notebook_outputs/tables/sec3a_institutions_map.csv")

df = pd.read_csv(SRC)

df["latitude"]  = pd.to_numeric(df.get("latitude"), errors="coerce")
df["longitude"] = pd.to_numeric(df.get("longitude"), errors="coerce")

def s_or_empty(x) -> str:
    return "" if pd.isna(x) else str(x)

# === ISO2 -> world_region (continenti)
# Nota: "Americas" è continent-level, poi separiamo North/South più avanti
ISO2_TO_WORLD = {
    # Europe
    "CZ":"Europe","DE":"Europe","AT":"Europe","CH":"Europe","FR":"Europe","IT":"Europe","ES":"Europe","PT":"Europe",
    "NL":"Europe","BE":"Europe","SE":"Europe","NO":"Europe","FI":"Europe","DK":"Europe","UK":"Europe","IE":"Europe",
    "PL":"Europe","RO":"Europe","HU":"Europe","GR":"Europe","TR":"Europe","UA":"Europe","MD":"Europe",
    # Asia
    "KR":"Asia","KP":"Asia","CN":"Asia","JP":"Asia","TW":"Asia","HK":"Asia","MO":"Asia","SG":"Asia","IN":"Asia",
    "TH":"Asia","VN":"Asia","MY":"Asia","ID":"Asia","PH":"Asia","IL":"Asia","SA":"Asia","AE":"Asia","QA":"Asia",
    "IR":"Asia","IQ":"Asia","PK":"Asia","BD":"Asia","LK":"Asia","NP":"Asia",
    # Americas
    "US":"Americas","CA":"Americas","MX":"Americas","BR":"Americas","AR":"Americas","CL":"Americas","CO":"Americas",
    "PE":"Americas","VE":"Americas","EC":"Americas","BO":"Americas","PY":"Americas","UY":"Americas","GY":"Americas",
    "SR":"Americas","GF":"Americas",
    # Oceania
    "AU":"Oceania","NZ":"Oceania",
    # Africa
    "ZA":"Africa","EG":"Africa","NG":"Africa","KE":"Africa","MA":"Africa","TN":"Africa","DZ":"Africa","GH":"Africa",
}

# 1) riempi world_region se manca usando country_code
mask_wr_missing = df["world_region"].isna() | (df["world_region"].astype(str).str.strip() == "")
cc = df["country_code"].apply(lambda x: s_or_empty(x).strip().upper())
df.loc[mask_wr_missing, "world_region"] = cc.loc[mask_wr_missing].map(ISO2_TO_WORLD)

# 2) fallback: coordinate per quello che resta
mask_wr_still = df["world_region"].isna() | (df["world_region"].astype(str).str.strip() == "")

def infer_world_region_from_coords(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return None
    lat = float(lat); lon = float(lon)
    if lon < -30:
        return "Americas"
    if -30 <= lon < 60:
        return "Europe" if lat >= 35 else "Africa"
    if lon >= 60:
        if lat < 0 and lon >= 110:
            return "Oceania"
        return "Asia"
    return None

df.loc[mask_wr_still, "world_region"] = df.loc[mask_wr_still].apply(
    lambda r: infer_world_region_from_coords(r.get("latitude"), r.get("longitude")),
    axis=1
)

# 3) region_final: split Americas in North/South
NORTH_AMERICA = {
    "US","CA","MX","GT","BZ","SV","HN","NI","CR","PA","CU","DO","HT","JM","BS","BB","TT","GD","LC","VC","AG","DM","KN"
}
SOUTH_AMERICA = {"BR","AR","CL","CO","PE","VE","EC","BO","PY","UY","GY","SR","GF"}

def map_region_final(row) -> str:
    wr = s_or_empty(row.get("world_region")).strip()
    cc = s_or_empty(row.get("country_code")).strip().upper()
    lat = row.get("latitude")

    if wr == "Americas":
        if cc in NORTH_AMERICA:
            return "North America"
        if cc in SOUTH_AMERICA:
            return "South America"
        if pd.notna(lat):
            return "South America" if float(lat) < 15 else "North America"
        return "North America"

    if wr in {"Europe", "Asia", "Oceania", "Africa"}:
        return wr

    return "Other"

df["region_final"] = df.apply(map_region_final, axis=1)

print("world_region counts:")
print(df["world_region"].value_counts(dropna=False))
print("\nregion_final counts:")
print(df["region_final"].value_counts(dropna=False))

OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False)
print("\n✅ Saved:", OUT)


world_region counts:
world_region
Americas    512
Europe      443
Asia        185
Oceania      19
Africa        2
Name: count, dtype: int64

region_final counts:
region_final
North America    495
Europe           443
Asia             185
Oceania           19
South America     17
Africa             2
Name: count, dtype: int64

✅ Saved: ../data/processed/outputs/openalex_notebook_outputs/tables/sec3a_institutions_map.csv


### institutionsMapData

In [12]:
import pandas as pd
import json
from pathlib import Path

# =========================
# INPUT / OUTPUT
# =========================
SRC = Path("../data/processed/outputs/openalex_notebook_outputs/tables/sec3a_institutions_map.csv")
OUT_JS = Path("../site/data/research/institutionsMapData.js")

TOP_N = None  # 20 / 100 / None per tutte

# =========================
# Helpers (NaN-safe)
# =========================
def s_or_empty(x) -> str:
    return "" if pd.isna(x) else str(x)

# =========================
# REGION mapping
# =========================
NORTH_AMERICA = {
    "US","CA","MX","GT","BZ","SV","HN","NI","CR","PA","CU","DO","HT","JM","BS","BB","TT","GD","LC","VC","AG","DM","KN"
}
SOUTH_AMERICA = {
    "BR","AR","CL","CO","PE","VE","EC","BO","PY","UY","GY","SR","GF"
}

def map_region(row) -> str:
    wr = s_or_empty(row.get("world_region")).strip()
    cc = s_or_empty(row.get("country_code")).strip().upper()
    lat = row.get("latitude")

    if wr == "Americas":
        if cc in NORTH_AMERICA:
            return "North America"
        if cc in SOUTH_AMERICA:
            return "South America"
        # fallback con lat (Centro America / unknown)
        if pd.notna(lat):
            try:
                return "South America" if float(lat) < 15 else "North America"
            except Exception:
                pass
        return "North America"

    if wr in {"Europe", "Asia", "Oceania", "Africa"}:
        return wr

    return "Other"

# =========================
# LOAD + clean
# =========================
df = pd.read_csv(SRC)

# numerici
df["latitude"] = pd.to_numeric(df.get("latitude"), errors="coerce")
df["longitude"] = pd.to_numeric(df.get("longitude"), errors="coerce")
df["n_papers_dataset"] = pd.to_numeric(df.get("n_papers_dataset"), errors="coerce").fillna(0).astype(int)

# tieni solo righe con coordinate
df = df.dropna(subset=["latitude", "longitude"]).copy()

# region finale
df["region_final"] = df.apply(map_region, axis=1)

# city label (NaN-safe)
def city_label(row) -> str:
    city = s_or_empty(row.get("city")).strip()
    reg  = s_or_empty(row.get("region")).strip()  # state/province se presente
    if city and reg:
        return f"{city}, {reg}"
    return city or ""

df["city_label"] = df.apply(city_label, axis=1)

# =========================
# TOP N (per n_papers_dataset)
# =========================
df = df.sort_values("n_papers_dataset", ascending=False)
if TOP_N is not None:
    df = df.head(int(TOP_N)).copy()

# =========================
# Nomi univoci (se duplicati)
# =========================
# (display_name può avere NaN: rendiamolo stringa safe)
df["display_name_safe"] = df["display_name"].apply(lambda x: s_or_empty(x).strip() or "Unknown Institution")
name_counts = df["display_name_safe"].value_counts().to_dict()

def unique_name(row) -> str:
    name = s_or_empty(row.get("display_name_safe")).strip() or "Unknown Institution"
    if name_counts.get(name, 0) > 1:
        cn = s_or_empty(row.get("country_name")).strip()
        iid = s_or_empty(row.get("institution_id")).strip()
        if cn:
            return f"{name} ({cn})"
        if iid:
            return f"{name} ({iid})"
    return name

df["name_unique"] = df.apply(unique_name, axis=1)

# =========================
# Build institutionsMapData
# =========================
institutions = []
for _, r in df.iterrows():
    institutions.append({
        "name": r["name_unique"],
        "country": s_or_empty(r.get("country_name")).strip(),
        "lat": float(r["latitude"]),
        "lon": float(r["longitude"]),
        "papers": int(r["n_papers_dataset"]),
        "region": r["region_final"],
        "city": r["city_label"],
    })

# ordina per regione poi papers desc
region_order = {
    "North America": 0, "Europe": 1, "Asia": 2,
    "Oceania": 3, "South America": 4, "Africa": 5,
    "Other": 6
}
institutions = sorted(
    institutions,
    key=lambda x: (region_order.get(x["region"], 99), -x["papers"], x["name"])
)

# =========================
# Stats
# =========================
total_papers = sum(x["papers"] for x in institutions)
regions_counts = {}
for x in institutions:
    regions_counts[x["region"]] = regions_counts.get(x["region"], 0) + 1

top_inst = max(institutions, key=lambda x: x["papers"]) if institutions else None

institutionsMapStats = {
    "totalInstitutions": int(len(institutions)),
    "totalPapers": int(total_papers),
    "regions": regions_counts,
    "topInstitution": (top_inst["name"] if top_inst else None),
    "topInstitutionPapers": (top_inst["papers"] if top_inst else 0),
}

regionColors = {
    "North America": "#3b82f6",
    "Europe": "#10b981",
    "Asia": "#f59e0b",
    "Oceania": "#8b5cf6",
    "South America": "#ef4444",
    "Africa": "#ec4899",
    "Other": "#9ca3af",
}

# =========================
# WRITE JS (newlines vere)
# =========================
OUT_JS.parent.mkdir(parents=True, exist_ok=True)

lines = [
    "/**",
    " * data/research/institutionsMapData.js",
    " * Geographic data for institutions bubble map",
    " * AUTO-GENERATED from sec3a_institutions_map.csv",
    " */",
    "",
    "export const institutionsMapData = " + json.dumps(institutions, ensure_ascii=False, indent=2) + ";",
    "",
    "export const institutionsMapStats = " + json.dumps(institutionsMapStats, ensure_ascii=False, indent=2) + ";",
    "",
    "export const regionColors = " + json.dumps(regionColors, ensure_ascii=False, indent=2) + ";",
    "",
]

OUT_JS.write_text("\n".join(lines), encoding="utf-8")
print("✅ Wrote:", OUT_JS.resolve())
print("Top institution:", institutionsMapStats["topInstitution"], institutionsMapStats["topInstitutionPapers"])
print("Regions:", institutionsMapStats["regions"])


✅ Wrote: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/site/data/research/institutionsMapData.js
Top institution: University of Utah 146
Regions: {'North America': 495, 'Europe': 443, 'Asia': 185, 'Oceania': 19, 'South America': 17, 'Africa': 2}


### institutionsCollaborationData

In [15]:
# =========================
# INPUT / OUTPUT
# =========================
IN_MAP   = Path("../data/processed/outputs/openalex_notebook_outputs/tables/sec3a_institutions_map.csv")
IN_EDGES = Path("../data/processed/outputs/openalex_notebook_outputs/tables/institution_edges.csv")

OUT_JS   = Path("../site/data/research/institutionsCollaborationData.js")

TOP_N = 20              # change this if you need 
SELECT_MODE = "papers"  # "papers" oppure "collab" (vedi sotto)


# =========================
# Region colors (puoi aggiungere South America/Africa/Other)
# =========================
regionColorsChord = {
    "North America": "#3b82f6",
    "Europe": "#10b981",
    "Asia": "#f59e0b",
    "Oceania": "#8b5cf6",
    "South America": "#ef4444",
    "Africa": "#ec4899",
    "Other": "#9ca3af",
}

def safe_str(x) -> str:
    return "" if pd.isna(x) else str(x)

def short_label(name: str) -> str:
    name = (name or "").strip()
    if not name:
        return name

    # Harvard University -> Harvard
    if name.endswith(" University") and not name.startswith("University of "):
        return name.replace(" University", "").strip()

    # University of Utah -> Utah (o "U. Utah" se preferisci)
    if name.startswith("University of "):
        rest = name.replace("University of ", "").strip()
        # University of California, Davis -> UC Davis
        if rest.startswith("California, "):
            return "UC " + rest.replace("California, ", "").strip()
        return rest

    return name

# =========================
# LOAD
# =========================
inst = pd.read_csv(IN_MAP)
edges = pd.read_csv(IN_EDGES)

# colonne minime richieste
need_cols = {"institution_id","display_name","region_final"}
missing = need_cols - set(inst.columns)
if missing:
    raise ValueError(f"sec3a_institutions_map.csv manca colonne: {missing}")

# normalizza
inst["institution_id"] = inst["institution_id"].astype(str)
inst["display_name"] = inst["display_name"].apply(lambda x: safe_str(x).strip())
inst["region_final"] = inst["region_final"].apply(lambda x: safe_str(x).strip() if safe_str(x).strip() else "Other")

# n_papers_dataset (per scegliere top)
if "n_papers_dataset" in inst.columns:
    inst["n_papers_dataset"] = pd.to_numeric(inst["n_papers_dataset"], errors="coerce").fillna(0).astype(int)
else:
    inst["n_papers_dataset"] = 0

edges["institution_id_1"] = edges["institution_id_1"].astype(str)
edges["institution_id_2"] = edges["institution_id_2"].astype(str)
edges["weight"] = pd.to_numeric(edges["weight"], errors="coerce").fillna(0).astype(int)

# =========================
# SELECT TOP institutions
# =========================
if SELECT_MODE == "papers":
    # top per numero paper nel tuo dataset
    selected = inst.sort_values("n_papers_dataset", ascending=False).head(TOP_N).copy()

elif SELECT_MODE == "collab":
    # top per "forza collaborazione" (somma pesi su tutti i vicini)
    # costruisci total weight per istituzione da edges
    w1 = edges.groupby("institution_id_1")["weight"].sum()
    w2 = edges.groupby("institution_id_2")["weight"].sum()
    totals = (w1.add(w2, fill_value=0)).reset_index()
    totals.columns = ["institution_id","collab_weight_sum"]
    inst2 = inst.merge(totals, on="institution_id", how="left")
    inst2["collab_weight_sum"] = inst2["collab_weight_sum"].fillna(0)
    selected = inst2.sort_values("collab_weight_sum", ascending=False).head(TOP_N).copy()

else:
    raise ValueError("SELECT_MODE deve essere 'papers' oppure 'collab'")

selected_ids = set(selected["institution_id"].tolist())

# =========================
# Build labels + ensure uniqueness
# =========================
selected["label"] = selected["display_name"].apply(short_label)

# se dopo lo shortening ci sono duplicati, disambigua
dups = selected["label"].value_counts()
dup_labels = set(dups[dups > 1].index.tolist())
if dup_labels:
    def disambig(row):
        lab = row["label"]
        if lab in dup_labels:
            cc = safe_str(row.get("country_code")).strip()
            if cc:
                return f"{lab} ({cc})"
            return f"{lab} ({row['institution_id']})"
        return lab
    selected["label"] = selected.apply(disambig, axis=1)

# ordine finale: (stabile) per papers desc, poi label
selected = selected.sort_values(["n_papers_dataset","label"], ascending=[False, True]).reset_index(drop=True)

institutions = selected["label"].tolist()
institutionRegions = selected["region_final"].tolist()

id_to_index = {iid: i for i, iid in enumerate(selected["institution_id"].tolist())}

# =========================
# Filter edges among selected + build matrix
# =========================
edges_sel = edges[
    edges["institution_id_1"].isin(selected_ids) &
    edges["institution_id_2"].isin(selected_ids) &
    (edges["weight"] > 0)
].copy()

n = len(institutions)
matrix = np.zeros((n, n), dtype=int)

for _, r in edges_sel.iterrows():
    a = r["institution_id_1"]
    b = r["institution_id_2"]
    w = int(r["weight"])
    i = id_to_index.get(a)
    j = id_to_index.get(b)
    if i is None or j is None or i == j:
        continue
    matrix[i, j] += w
    matrix[j, i] += w  # simmetrica

# diagonale a zero (sicurezza)
np.fill_diagonal(matrix, 0)

# =========================
# Stats
# =========================
# totalCollaborations = somma triangolo superiore
totalCollaborations = int(np.triu(matrix, 1).sum())

# avgCollaborationsPerInstitution: media della somma collaborazioni per istituzione
# (equivale a 2*totalCollaborations / N)
row_sums = matrix.sum(axis=1)
avgCollab = float(row_sums.mean()) if n else 0.0

# strongest pair
strongestPair = {"inst1": None, "inst2": None, "papers": 0}
if n > 1 and totalCollaborations > 0:
    tri = np.triu(matrix, 1)
    idx = np.unravel_index(np.argmax(tri), tri.shape)
    i, j = int(idx[0]), int(idx[1])
    strongestPair = {"inst1": institutions[i], "inst2": institutions[j], "papers": int(tri[i, j])}

# regional clusters
regionalClusters = {}
for name, reg in zip(institutions, institutionRegions):
    reg = reg or "Other"
    regionalClusters.setdefault(reg, []).append(name)

# crossRegionalRate
cross_sum = 0
for i in range(n):
    for j in range(i+1, n):
        if matrix[i, j] > 0 and institutionRegions[i] != institutionRegions[j]:
            cross_sum += matrix[i, j]
crossRegionalRate = float(cross_sum / totalCollaborations) if totalCollaborations else 0.0

institutionsCollaborationStats = {
    "totalInstitutions": int(n),
    "totalCollaborations": int(totalCollaborations),
    "avgCollaborationsPerInstitution": float(round(avgCollab, 2)),
    "strongestPair": strongestPair,
    "regionalClusters": regionalClusters,
    "crossRegionalRate": float(round(crossRegionalRate, 4)),
}

# =========================
# Build JS object (mockup-compatible)
# =========================
institutionsCollaborationData = {
    "institutions": institutions,
    "institutionRegions": institutionRegions,
    "matrix": matrix.tolist(),
}

# =========================
# WRITE JS (newlines vere)
# =========================
OUT_JS.parent.mkdir(parents=True, exist_ok=True)

lines = [
    "/**",
    " * data/research/institutionsCollaborationData.js",
    " * Chord diagram data for inter-institutional collaborations",
    " * AUTO-GENERATED from institution_edges.csv + sec3a_institutions_map.csv",
    " */",
    "",
    "export const institutionsCollaborationData = " + json.dumps(institutionsCollaborationData, ensure_ascii=False, indent=2) + ";",
    "",
    "export const institutionsCollaborationStats = " + json.dumps(institutionsCollaborationStats, ensure_ascii=False, indent=2) + ";",
    "",
    "export const regionColorsChord = " + json.dumps(regionColorsChord, ensure_ascii=False, indent=2) + ";",
    "",
]

OUT_JS.write_text("\n".join(lines), encoding="utf-8")

print("✅ Wrote:", OUT_JS.resolve())
print("Institutions:", n)
print("Total collaborations (upper triangle):", totalCollaborations)
print("Strongest pair:", strongestPair)
print("Region counts:", pd.Series(institutionRegions).value_counts().to_dict())


✅ Wrote: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/site/data/research/institutionsCollaborationData.js
Institutions: 20
Total collaborations (upper triangle): 306
Strongest pair: {'inst1': 'Hong Kong University of Science and Technology', 'inst2': 'Hong Kong', 'papers': 98}
Region counts: {'North America': 12, 'Europe': 5, 'Asia': 3}
